# Real Baseline: TimeDiff on **eICU** (MuhangTian/TimeDiff, JAMIA 2024)


## 1 · Clone TimeDiff and install dependencies

In [ ]:
import os, sys, subprocess
if not os.path.exists("/content/TimeDiff"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/MuhangTian/TimeDiff.git", "/content/TimeDiff"], check=True)
sys.path.insert(0, "/content/TimeDiff")
os.environ["PYTHONPATH"] = "/content/TimeDiff"
print("cloned TimeDiff:", os.listdir("/content/TimeDiff")[:12])

In [ ]:
!pip -q install ema-pytorch einops accelerate torchcde 2>/dev/null | tail -2
print("deps installed (ignore resolver warnings)")

## 2 · Setup

In [ ]:
import time, numpy as np, torch
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)
assert device == "cuda", "Enable a GPU runtime — TimeDiff training needs it."
from google.colab import drive
drive.mount('/content/drive')

## 3 · Inspect the real TimeDiff source

In [ ]:
import inspect
from models.ETDiff.utils import TimeSeriesDataset
from models.ETDiff.mixed_diffusion import MixedDiffusion
from models.ETDiff.blocks import RNN
print("=" * 30, "TimeSeriesDataset", "=" * 30)
print(inspect.getsource(TimeSeriesDataset)[:2500])
print("=" * 30, "MixedDiffusion.__init__", "=" * 30)
print(inspect.getsource(MixedDiffusion.__init__)[:1500])
try:
    print("=" * 30, "MixedDiffusion.sample", "=" * 30)
    print(inspect.getsource(MixedDiffusion.sample)[:1200])
except Exception as e:
    print("(sample signature not found:", e, ")")

## 4 · Configuration

In [ ]:
V2_CACHE = "..."
TD_DIR   = "..."; os.makedirs(TD_DIR, exist_ok=True)
TASK       = "mortality_inhosp"
TEST_FRAC  = 0.20
TD_STEPS   = 400000
TD_BATCH   = 32
TD_TIMESTEPS = 1000
N_SAMPLE   = 8000
WINS = (0.005, 0.995)
SEED = 0
rng = np.random.default_rng(SEED)
print(f"TimeDiff Phase C | steps {TD_STEPS} | task {TASK}")

In [ ]:
OUTDIR = "/content/drive/MyDrive/eICU/eICU_TimeDiff_results"
os.makedirs(OUTDIR, exist_ok=True)

## 5 · cohort, and the TimeDiff data adapter

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score


# featurizers
def mask_features(m):
    """m:(N,T,V) {0,1} -> (N,4V): count, rate, first-obs-time, last-obs-time."""
    N, T, V = m.shape
    count = m.sum(1).astype(np.float32)
    rate = count / T
    obs = m > 0.5
    first = np.where(obs.any(1), np.argmax(obs, 1), T).astype(np.float32) / T
    rev = obs[:, ::-1, :]
    last_idx = np.where(obs.any(1), (T - 1 - np.argmax(rev, 1)), -1.0).astype(np.float32)
    last = (last_idx + 1.0) / T
    return np.concatenate([count, rate, first, last], axis=1).astype(np.float32)


def value_features(y, m):
    """Per-variable summary over OBSERVED entries: mean, std, min, max, last value."""
    N, T, V = y.shape
    obs = m > 0.5
    cnt = np.maximum(m.sum(1), 1)
    mean = (y.sum(1) / cnt).astype(np.float32)
    var = np.maximum((y * y).sum(1) / cnt - mean ** 2, 0.0)
    std = np.sqrt(var).astype(np.float32)
    any_obs = obs.any(1)
    vmin = np.where(any_obs, np.where(obs, y, np.inf).min(1), 0.0).astype(np.float32)
    vmax = np.where(any_obs, np.where(obs, y, -np.inf).max(1), 0.0).astype(np.float32)
    rev = obs[:, ::-1, :]
    last_t = np.where(any_obs, T - 1 - np.argmax(rev, 1), 0)
    last = np.take_along_axis(y, last_t[:, None, :], axis=1)[:, 0, :]
    last = np.where(any_obs, last, 0.0).astype(np.float32)
    feats = np.concatenate([mean, std, vmin, vmax, last], axis=1)
    return np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)


def channel_features(m, y, channel):
    """channel in {mask, value, joint}."""
    parts = []
    if channel in ("mask", "joint"):
        parts.append(mask_features(m))
    if channel in ("value", "joint"):
        parts.append(value_features(y, m))
    return np.nan_to_num(np.concatenate(parts, axis=1))


#  TSTR
def _clf(seed):
    return HistGradientBoostingClassifier(max_depth=4, max_iter=200, learning_rate=0.1,
                                          random_state=seed)


def transfer_auroc(m_tr, y_tr, lab_tr, m_te, y_te, lab_te, channel, seed=0, n_boot=300):
    """Train a predictor on (train) features->label, evaluate AUROC on (test).
    TSTR: train=synthetic, test=real. TRTR: train=real, test=real. Bootstrap CI over test."""
    Xtr = channel_features(m_tr, y_tr, channel)
    Xte = channel_features(m_te, y_te, channel)
    sc = StandardScaler().fit(Xtr)
    clf = _clf(seed).fit(sc.transform(Xtr), lab_tr)
    p = clf.predict_proba(sc.transform(Xte))[:, 1]
    auc = roc_auc_score(lab_te, p)
    rng = np.random.default_rng(seed); n = len(lab_te); boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(lab_te[idx])) == 2:
            boots.append(roc_auc_score(lab_te[idx], p[idx]))
    lo, hi = (np.percentile(boots, [2.5, 97.5]) if boots else (auc, auc))
    return float(auc), float(lo), float(hi)


In [ ]:
d = np.load(V2_CACHE, allow_pickle=True)
m_all, y_all = d["m"].astype(np.float32), d["y"].astype(np.float32)
VAR_NAMES = list(d["var_names"]); VAR_CLASS = [str(x) for x in d["var_class"]]
N, T, V = m_all.shape
labels = d[f"lab_{TASK}"].astype(int)
tr_idx, te_idx = train_test_split(np.arange(N), test_size=TEST_FRAC, stratify=labels, random_state=SEED)

lo_hi = np.zeros((V, 2), np.float32)
y_w = y_all.copy()
for v in range(V):
    obs = y_all[tr_idx][:, :, v][m_all[tr_idx][:, :, v] > 0.5]
    if obs.size > 200:
        lo, hi = np.quantile(obs, WINS); lo_hi[v] = (lo, hi)
        y_w[:, :, v] = np.clip(y_w[:, :, v], lo, hi) * m_all[:, :, v]
    else:
        lo_hi[v] = (0.0, 1.0)
m_tr, y_tr = m_all[tr_idx], y_w[tr_idx]
m_te, y_te = m_all[te_idx], y_w[te_idx]
lab_tr, lab_te = labels[tr_idx], labels[te_idx]
print(f"train {len(tr_idx)} | test {len(te_idx)} | V={V} T={T} | prevalence {lab_tr.mean():.3f}")

In [ ]:
def to_unit(y, v):
    lo, hi = lo_hi[v]; rng_ = max(hi - lo, 1e-6)
    return (2.0 * (y - lo) / rng_ - 1.0)
C = 2 * V + 1
X_td = np.zeros((len(tr_idx), T, C), np.float32)
for v in range(V):
    X_td[:, :, 2 * v]     = np.where(m_tr[:, :, v] > 0.5, to_unit(y_tr[:, :, v], v), 0.0)
    X_td[:, :, 2 * v + 1] = (m_tr[:, :, v] > 0.5).astype(np.float32)
X_td[:, :, -1] = lab_tr[:, None].astype(np.float32)

numerical_idx   = [2 * v for v in range(V)]
categorical_idx = [2 * v + 1 for v in range(V)] + [C - 1]
num_classes     = [2] * (V + 1)
os.makedirs(f"{TD_DIR}/data", exist_ok=True)
X_td_cf = np.transpose(X_td, (0, 2, 1))
torch.save(torch.from_numpy(np.ascontiguousarray(X_td_cf)), f"{TD_DIR}/data/train.pt")
print("saved as (N, C, T):", X_td_cf.shape)
print(f"TimeDiff tensor: {X_td.shape} (N,T,C) | numerical {len(numerical_idx)} | categorical {len(categorical_idx)}")

## 6 · Build TimeDiff (ETDiff + MixedDiffusion + RNN), as in `etdiff_train.py`

In [ ]:
from models.ETDiff.mixed_diffusion import MixedDiffusion
from models.ETDiff.et_diff import ETDiff
from models.ETDiff.blocks import RNN
from models.ETDiff.utils import TimeSeriesDataset

os.chdir("/content/TimeDiff")
os.makedirs("data/datasets/eicu_inhosp", exist_ok=True)

dataset = TimeSeriesDataset(categorical_cols=categorical_idx, data_name="eicu_inhosp",
                            load_path=f"{TD_DIR}/data/train.pt", cut_length=T)
print("dataset channels", getattr(dataset, "channels", "?"), "| seq_length", getattr(dataset, "seq_length", "?"))
print("dataset tensor shape:", tuple(dataset.data.shape) if hasattr(dataset, "data") else "no .data attr")

model = RNN(
    input_channels  = dataset.channels + len(dataset.categorical_cols),
    hidden_channels = (dataset.channels + len(dataset.categorical_cols)) * 4,
    output_channels = dataset.channels + len(dataset.categorical_cols),
    layers = 3, model = "lstm", dropout = 0.0, bidirectional = False,
    self_condition = False, embed_dim = 64, time_dim = 256,
)
diffusion = MixedDiffusion(
    model = model, channels = dataset.channels + len(dataset.categorical_cols),
    seq_length = dataset.seq_length, timesteps = TD_TIMESTEPS, auto_normalize = True,
    numerical_features_indices = numerical_idx,
    categorical_features_indices = categorical_idx,
    categorical_num_classes = num_classes, loss_lambda = 0.8,
)
etdiff = ETDiff(
    diffusion_model = diffusion, dataset = dataset, sample_every = 100000,
    train_batch_size = TD_BATCH, diff_lr = 8e-5, diff_num_steps = TD_STEPS,
    gradient_accumulate_every = 2, ema_decay = 0.995, amp = False, wandb = None,
    check_point_path = f"{TD_DIR}/model.pt", run_id = "phaseC",
)
print("ETDiff built |", sum(p.numel() for p in model.parameters() if p.requires_grad), "params")

## 7 · Train TimeDiff and generate synthetic records

In [ ]:
t0 = time.time()
etdiff.train()
print(f"TimeDiff trained in {(time.time()-t0)/60:.1f} min ({TD_STEPS} steps)")

@torch.no_grad()
def collect(model, n, bs=100):
    model.eval(); out = []
    for _ in range(max(1, n // bs)):
        out.append(model.sample(batch_size=bs))
    return torch.cat(out, 0).squeeze().cpu().numpy()
samples = collect(etdiff.ema.ema_model, N_SAMPLE)
if hasattr(etdiff, "sample_min") and etdiff.sample_min is not None:
    from helpers.utils import reverse_normalize
    samples = reverse_normalize(samples, etdiff.sample_min, etdiff.sample_max)
print("raw samples:", samples.shape, "(expect N,T,C or N,C,T — de-interleave next)")

## 8 · De-interleave samples into (mask, value, label)

In [ ]:
S = samples
if S.ndim == 3 and S.shape[1] == C and S.shape[2] == T:
    S = np.transpose(S, (0, 2, 1))
assert S.ndim == 3 and S.shape[2] == C, f"unexpected sample shape {S.shape}; inspect and adjust"

M_td = (S[:, :, [2 * v + 1 for v in range(V)]] > 0.5).astype(np.float32)     # masks
def from_unit(u, v):
    lo, hi = lo_hi[v]; return (u + 1.0) / 2.0 * max(hi - lo, 1e-6) + lo
X_val = np.zeros((S.shape[0], T, V), np.float32)
for v in range(V):
    X_val[:, :, v] = np.clip(from_unit(S[:, :, 2 * v], v), lo_hi[v, 0], lo_hi[v, 1])
Y_td = (M_td * X_val).astype(np.float32)
lab_td = (np.round(S[:, :, -1].mean(1)) > 0.5).astype(int)                   # per-patient generated label
print(f"TimeDiff synthetic: masks {M_td.shape} | mask density {M_td.mean():.3f} | "
      f"generated prevalence {lab_td.mean():.3f} (real {lab_tr.mean():.3f})")

## 9 · Channel-decomposition TSTR: TimeDiff vs the ceiling

In [ ]:
RECORDDIFF_C = {"mask": 0.690, "value": 0.755, "joint": 0.762}   # eICU Phase C in-hospital
print(f"{'channel':7s} | {'TRTR (ceiling)':>16s} {'TimeDiff TSTR':>16s} {'RecordDiff TSTR':>16s}")
print("-" * 62)
rows = {}
for ch in ["mask", "value", "joint"]:
    trtr = transfer_auroc(m_tr, y_tr, lab_tr, m_te, y_te, lab_te, ch, seed=SEED)
    td = transfer_auroc(M_td, Y_td, lab_td, m_te, y_te, lab_te, ch, seed=SEED)
    rows[ch] = (trtr, td)
    print(f"{ch:7s} | {trtr[0]:>7.3f} [{trtr[1]:.2f},{trtr[2]:.2f}] {td[0]:>7.3f} [{td[1]:.2f},{td[2]:.2f}]"
          f" {RECORDDIFF_C[ch]:>16.3f}")

chans = ["mask", "value", "joint"]; x = np.arange(3); w = 0.26
fig = plt.figure(figsize=(8.5, 5))
plt.bar(x - w, [rows[c][0][0] for c in chans], w, label="TRTR (ceiling)", color="0.6")
plt.bar(x,     [RECORDDIFF_C[c] for c in chans], w, label="RecordDiff", color="teal")
plt.bar(x + w, [rows[c][1][0] for c in chans], w, label="TimeDiff (released)", color="purple")
plt.axhline(0.5, ls="--", c="k"); plt.xticks(x, chans); plt.ylim(0.45, 1.0)
plt.ylabel("AUROC (test on real)"); plt.title(f"Channel-decomposition TSTR — {TASK}")
plt.legend(); plt.tight_layout()

fname = f"channel_TSTR_{TASK}".replace(" ", "_").replace("/", "-")
png = os.path.join(OUTDIR, fname + ".png")
pdf = os.path.join(OUTDIR, fname + ".pdf")
plt.savefig(png, dpi=300, bbox_inches="tight")
plt.savefig(pdf, bbox_inches="tight")
plt.show()
plt.close(fig)

print("saved:", png)

In [ ]:
TD_DATA_NAME_ICU = "eicu_icu"
os.chdir("/content/TimeDiff")
os.makedirs(f"data/datasets/{TD_DATA_NAME_ICU}", exist_ok=True)
os.makedirs(f"{TD_DIR}/data_icu", exist_ok=True)

labels_icu_all = np.load(V2_CACHE, allow_pickle=True)["lab_icu_mortality"].astype(int)
lab_tr_icu, lab_te_icu = labels_icu_all[tr_idx], labels_icu_all[te_idx]

def to_unit(y, v):
    lo, hi = lo_hi[v]; return 2.0 * (y - lo) / max(hi - lo, 1e-6) - 1.0

X_icu = np.zeros((len(tr_idx), T, C), np.float32)
for v in range(V):
    X_icu[:, :, 2 * v]     = np.where(m_tr[:, :, v] > 0.5, to_unit(y_tr[:, :, v], v), 0.0)
    X_icu[:, :, 2 * v + 1] = (m_tr[:, :, v] > 0.5).astype(np.float32)
X_icu[:, :, -1] = lab_tr_icu[:, None].astype(np.float32)
X_icu_cf = np.ascontiguousarray(np.transpose(X_icu, (0, 2, 1)))
torch.save(torch.from_numpy(X_icu_cf), f"{TD_DIR}/data_icu/train.pt")
print(f"ICU training tensor {X_icu_cf.shape} (N,C,T) | ICU prevalence {lab_tr_icu.mean():.3f}")

In [ ]:
from models.ETDiff.mixed_diffusion import MixedDiffusion
from models.ETDiff.et_diff import ETDiff
from models.ETDiff.blocks import RNN
from models.ETDiff.utils import TimeSeriesDataset

dataset_icu = TimeSeriesDataset(categorical_cols=categorical_idx, data_name=TD_DATA_NAME_ICU,
                                load_path=f"{TD_DIR}/data_icu/train.pt", cut_length=T)
print("ICU dataset channels", getattr(dataset_icu, "channels", "?"),
      "| seq_length", getattr(dataset_icu, "seq_length", "?"))

model_icu = RNN(
    input_channels  = dataset_icu.channels + len(dataset_icu.categorical_cols),
    hidden_channels = (dataset_icu.channels + len(dataset_icu.categorical_cols)) * 4,
    output_channels = dataset_icu.channels + len(dataset_icu.categorical_cols),
    layers = 3, model = "lstm", dropout = 0.0, bidirectional = False,
    self_condition = False, embed_dim = 64, time_dim = 256,
)
diffusion_icu = MixedDiffusion(
    model = model_icu, channels = dataset_icu.channels + len(dataset_icu.categorical_cols),
    seq_length = dataset_icu.seq_length, timesteps = TD_TIMESTEPS, auto_normalize = True,
    numerical_features_indices = numerical_idx,
    categorical_features_indices = categorical_idx,
    categorical_num_classes = num_classes, loss_lambda = 0.8,
)
etdiff_icu = ETDiff(
    diffusion_model = diffusion_icu, dataset = dataset_icu, sample_every = 100000,
    train_batch_size = TD_BATCH, diff_lr = 8e-5, diff_num_steps = TD_STEPS,
    gradient_accumulate_every = 2, ema_decay = 0.995, amp = False, wandb = None,
    check_point_path = f"{TD_DIR}/model_icu.pt", run_id = "phaseC_icu",
)
t0 = time.time()
etdiff_icu.train()
print(f"TimeDiff (ICU) trained in {(time.time()-t0)/60:.1f} min ({TD_STEPS} steps)")

In [ ]:
@torch.no_grad()
def collect(model, n, bs=100):
    model.eval(); out = []
    for _ in range(max(1, n // bs)):
        out.append(model.sample(batch_size=bs))
    return torch.cat(out, 0).squeeze().cpu().numpy()

samples_icu = collect(etdiff_icu.ema.ema_model, N_SAMPLE)
if hasattr(etdiff_icu, "sample_min") and etdiff_icu.sample_min is not None:
    from helpers.utils import reverse_normalize
    samples_icu = reverse_normalize(samples_icu, etdiff_icu.sample_min, etdiff_icu.sample_max)

S = samples_icu
if S.ndim == 3 and S.shape[1] == C:
    S = np.transpose(S, (0, 2, 1))
assert S.ndim == 3 and S.shape[2] == C, f"unexpected sample shape {S.shape}"

def from_unit(u, v):
    lo, hi = lo_hi[v]; return (u + 1.0) / 2.0 * max(hi - lo, 1e-6) + lo

M_td_icu = (S[:, :, [2 * v + 1 for v in range(V)]] > 0.5).astype(np.float32)
Xv = np.zeros((S.shape[0], T, V), np.float32)
for v in range(V):
    Xv[:, :, v] = np.clip(from_unit(S[:, :, 2 * v], v), lo_hi[v, 0], lo_hi[v, 1])
Y_td_icu = (M_td_icu * Xv).astype(np.float32)
lab_td_icu = (np.round(S[:, :, -1].mean(1)) > 0.5).astype(int)
print(f"TimeDiff ICU synthetic: mask density {M_td_icu.mean():.3f} | "
      f"gen prevalence {lab_td_icu.mean():.3f} (real {lab_tr_icu.mean():.3f})")

In [ ]:
import os, io, json, contextlib, datetime
class Tee(io.StringIO):
    """Writes to the notebook AND keeps a copy in the buffer."""
    def write(self, s):
        import sys; sys.__stdout__.write(s)
        return super().write(s)

In [ ]:
# channel-decomposition TSTR on ICU mortality
RECORDDIFF_ICU = {"mask": 0.720, "value": 0.786, "joint": 0.795}
print(f"{'channel':7s} | {'TRTR (ceiling)':>16s} {'TimeDiff TSTR':>16s} {'RecordDiff TSTR':>16s}")
print("-" * 62)
rows_icu = {}
for ch in ["mask", "value", "joint"]:
    trtr = transfer_auroc(m_tr, y_tr, lab_tr_icu, m_te, y_te, lab_te_icu, ch, seed=SEED)
    td   = transfer_auroc(M_td_icu, Y_td_icu, lab_td_icu, m_te, y_te, lab_te_icu, ch, seed=SEED)
    rows_icu[ch] = (trtr, td)
    print(f"{ch:7s} | {trtr[0]:>7.3f} [{trtr[1]:.2f},{trtr[2]:.2f}] {td[0]:>7.3f} [{td[1]:.2f},{td[2]:.2f}]"
          f" {RECORDDIFF_ICU[ch]:>16.3f}")

chans = ["mask", "value", "joint"]; x = np.arange(3); w = 0.26
plt.figure(figsize=(8.5, 5))
plt.bar(x - w, [rows_icu[c][0][0] for c in chans], w, label="TRTR (ceiling)", color="0.6")
plt.bar(x,     [RECORDDIFF_ICU[c] for c in chans], w, label="RecordDiff", color="teal")
plt.bar(x + w, [rows_icu[c][1][0] for c in chans], w, label="TimeDiff (released)", color="purple")
plt.axhline(0.5, ls="--", c="k"); plt.xticks(x, chans); plt.ylim(0.45, 1.0)
plt.ylabel("AUROC (test on real)"); plt.title("Channel-decomposition TSTR — icu_mortality (label-matched)")
plt.legend(); plt.tight_layout(); plt.show()

try:
    print("\nPaste into the cross-dataset summary TIMEDIFF dict:")
    print('    "eICU": {')
    print(f'        "mortality_inhosp": {{"mask":{rows["mask"][1][0]:.3f},"value":{rows["value"][1][0]:.3f},"joint":{rows["joint"][1][0]:.3f}}},')
    print(f'        "icu_mortality":    {{"mask":{rows_icu["mask"][1][0]:.3f},"value":{rows_icu["value"][1][0]:.3f},"joint":{rows_icu["joint"][1][0]:.3f}}}}},')
except NameError:
    print("(run the in-hospital cells (through the RECORDDIFF_C table) first so `rows` exists)")